In [ ]:
# ===============================
# Cell 1: Import and Load Data
# ===============================
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.base import clone

print("📂 Loading data...")
df = pd.read_csv("../data/mcs_sh_sample_actig_Feb_28_preprocessed.csv")

df["suicide_17y"] = df["suicide_17y"].map({"no": 0, "yes": 1})

X = df.drop(columns=["suicide_17y"])
y = df["suicide_17y"].astype(int)

if "id" in X.columns:
    X = X.drop(columns=["id"])

X = X.replace({"True": True, "False": False})
X = X.astype(float)

print(X.dtypes.value_counts())

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print("✅ Data loaded and CV defined.")

# Define actigraphy and questionnaire columns
actigraphy_cols = [
    "mean_acc_24h",
    "m5_hour_start",
    "l5_hour_start",
    "mvpa_acc_5sec"
]
missing_actig = [c for c in actigraphy_cols if c not in X.columns]
if len(missing_actig) > 0:
    raise ValueError(f"These actigraphy cols are missing from X: {missing_actig}")

questionnaire_cols = [c for c in X.columns if c not in actigraphy_cols]

print(f"✅ Actigraphy cols: {len(actigraphy_cols)}")
print(f"✅ Questionnaire cols: {len(questionnaire_cols)}")

In [ ]:
# ===============================
# Cell 2: Two Stage Nested CV Evaluation
# ===============================
def find_best_threshold(y_true, y_probs, metric=f1_score):
    best_thresh = 0.5
    best_score = -1
    thresholds = np.linspace(0.0, 1.0, 101)
    for t in thresholds:
        preds = (y_probs >= t).astype(int)
        try:
            score = metric(y_true.astype(int), preds)
        except:
            continue
        if score > best_score:
            best_thresh = t
            best_score = score
    return best_thresh, best_score


def two_stage_nested_cv_evaluation(
    X, y,
    stage1_model, stage1_grid,
    stage2_model, stage2_grid,
    questionnaire_cols, actigraphy_cols,
    inner_cv, outer_cv,
    primary_metric="f1"
):

    def npv_score(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return tn / (tn + fn) if (tn + fn) > 0 else 0

    def specificity_score(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return tn / (tn + fp) if (tn + fp) > 0 else 0

    scoring_funcs = {
        "F1": f1_score,
        "Weighted F1": lambda y, p: f1_score(y, p, average="weighted"),
        "Precision": lambda y, p: precision_score(y, p, zero_division=0),
        "Recall": lambda y, p: recall_score(y, p, zero_division=0),
        "Specificity": specificity_score,
        "NPV": npv_score,
        "AUROC": roc_auc_score
    }

    fold_metrics_rows = []
    fold_predictions_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        print(f"\n🌊 Outer Fold {fold_idx}")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # ----------------------------
        # Stage 1: questionnaire model
        # ----------------------------
        Xq_train = X_train[questionnaire_cols]
        Xq_test = X_test[questionnaire_cols]

        gs1 = GridSearchCV(
            estimator=clone(stage1_model),
            param_grid=stage1_grid,
            cv=inner_cv,
            scoring=primary_metric,
            n_jobs=-1,
            verbose=0
        )
        gs1.fit(Xq_train, y_train)
        best_stage1 = gs1.best_estimator_

        # Use OOF predictions on outer-train to pick threshold and to define low risk
        p1_oof = cross_val_predict(
            clone(best_stage1),
            Xq_train,
            y_train,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        t1, _ = find_best_threshold(y_train.values, p1_oof)
        print(f"✅ Stage 1 threshold (OOF) fold {fold_idx}: {t1:.2f}")

        # Stage 1 on outer-test
        p1_test = best_stage1.predict_proba(Xq_test)[:, 1]
        yhat1_test = (p1_test >= t1).astype(int)

        # False negatives under stage 1 on this outer test
        fn_stage1_test = int(((y_test.values == 1) & (yhat1_test == 0)).sum())

        # Define low-risk group using OOF predictions on outer-train
        lowrisk_train_mask = p1_oof < t1

        # If low-risk train is too small or has no positive cases, skip stage 2
        y_train_low = y_train.iloc[lowrisk_train_mask]
        X_train_low_actig = X_train[actigraphy_cols].iloc[lowrisk_train_mask]

        can_run_stage2 = True
        if lowrisk_train_mask.sum() < 15:
            can_run_stage2 = False
            reason = "low-risk train set too small"
        elif y_train_low.sum() < 2:
            can_run_stage2 = False
            reason = "too few positives in low-risk train set"

        if not can_run_stage2:
            print(f"⚠️ Stage 2 skipped fold {fold_idx} because {reason}.")

            # Record stage 1 only metrics
            row1 = {"Fold": fold_idx, "Variant": "Stage 1 only", "Stage1_threshold": t1, "Stage2_threshold": np.nan}
            for metric, func in scoring_funcs.items():
                if metric == "AUROC":
                    row1[metric] = func(y_test.values, p1_test)
                else:
                    row1[metric] = func(y_test.values, yhat1_test)
            tn, fp, fn, tp = confusion_matrix(y_test.values, yhat1_test).ravel()
            row1.update({"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp), "Recovered_FNs": 0})
            fold_metrics_rows.append(row1)

            # Save per-sample predictions (stage 1 only)
            for j, idx in enumerate(test_idx):
                fold_predictions_rows.append({
                    "Fold": fold_idx,
                    "Index": int(idx),
                    "y_true": int(y.iloc[idx]),
                    "p_stage1": float(p1_test[j]),
                    "p_stage2": 0.0,
                    "pred_stage1": int(yhat1_test[j]),
                    "pred_two_stage": int(yhat1_test[j]),
                    "lowrisk_after_stage1": bool(p1_test[j] < t1)
                })
            continue

        # ----------------------------
        # Stage 2: actigraphy model
        # Runs only within low-risk subset of outer-train
        # ----------------------------
        gs2 = GridSearchCV(
            estimator=clone(stage2_model),
            param_grid=stage2_grid,
            cv=inner_cv,
            scoring=primary_metric,
            n_jobs=-1,
            verbose=0
        )
        gs2.fit(X_train_low_actig, y_train_low)
        best_stage2 = gs2.best_estimator_

        # Stage 2 threshold via OOF within low-risk train subset
        p2_oof = cross_val_predict(
            clone(best_stage2),
            X_train_low_actig,
            y_train_low,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        t2, _ = find_best_threshold(y_train_low.values, p2_oof)
        print(f"✅ Stage 2 threshold (OOF) fold {fold_idx}: {t2:.2f}")

        # Apply stage 2 on outer-test only for those low-risk per stage 1
        lowrisk_test_mask = p1_test < t1
        X_test_actig = X_test[actigraphy_cols]

        p2_test = np.zeros_like(p1_test)
        if lowrisk_test_mask.sum() > 0:
            p2_test[lowrisk_test_mask] = best_stage2.predict_proba(X_test_actig.iloc[lowrisk_test_mask])[:, 1]

        yhat2_test = (p2_test >= t2).astype(int)

        # Two-stage combined prediction
        yhat_two_stage = yhat1_test.copy()
        yhat_two_stage[lowrisk_test_mask] = yhat2_test[lowrisk_test_mask]

        # FN recovery count on this outer test
        recovered = int(((y_test.values == 1) & (yhat1_test == 0) & (yhat_two_stage == 1)).sum())

        # Stage 1 only metrics row
        row1 = {"Fold": fold_idx, "Variant": "Stage 1 only", "Stage1_threshold": t1, "Stage2_threshold": np.nan}
        for metric, func in scoring_funcs.items():
            if metric == "AUROC":
                row1[metric] = func(y_test.values, p1_test)
            else:
                row1[metric] = func(y_test.values, yhat1_test)
        tn, fp, fn, tp = confusion_matrix(y_test.values, yhat1_test).ravel()
        row1.update({"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp), "Recovered_FNs": 0})
        fold_metrics_rows.append(row1)

        # Two-stage metrics row
        # For AUROC, we use a combined score for ranking
        p_two_stage = np.maximum(p1_test, p2_test)

        row2 = {"Fold": fold_idx, "Variant": "Two stage", "Stage1_threshold": t1, "Stage2_threshold": t2}
        for metric, func in scoring_funcs.items():
            if metric == "AUROC":
                row2[metric] = func(y_test.values, p_two_stage)
            else:
                row2[metric] = func(y_test.values, yhat_two_stage)
        tn, fp, fn, tp = confusion_matrix(y_test.values, yhat_two_stage).ravel()
        row2.update({"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp), "Recovered_FNs": recovered})
        fold_metrics_rows.append(row2)

        print(f"📌 Fold {fold_idx} stage 1 FN on test: {fn_stage1_test}")
        print(f"📌 Fold {fold_idx} recovered FN with actigraphy: {recovered}")

        # Save per-sample predictions
        for j, idx in enumerate(test_idx):
            fold_predictions_rows.append({
                "Fold": fold_idx,
                "Index": int(idx),
                "y_true": int(y.iloc[idx]),
                "p_stage1": float(p1_test[j]),
                "p_stage2": float(p2_test[j]),
                "pred_stage1": int(yhat1_test[j]),
                "pred_two_stage": int(yhat_two_stage[j]),
                "lowrisk_after_stage1": bool(lowrisk_test_mask[j])
            })

    fold_metrics_df = pd.DataFrame(fold_metrics_rows)
    fold_predictions_df = pd.DataFrame(fold_predictions_rows)

    # Summary effect table
    summary = (
        fold_metrics_df
        .groupby("Variant")[["Recall", "Precision", "F1", "Specificity", "NPV", "AUROC", "FN", "Recovered_FNs"]]
        .mean()
        .reset_index()
    )

    # Add delta row (two stage minus stage 1)
    if set(summary["Variant"]) >= {"Stage 1 only", "Two stage"}:
        a = summary[summary["Variant"] == "Stage 1 only"].iloc[0]
        b = summary[summary["Variant"] == "Two stage"].iloc[0]
        delta = {"Variant": "Delta (Two stage - Stage 1)"}
        for col in ["Recall", "Precision", "F1", "Specificity", "NPV", "AUROC", "FN", "Recovered_FNs"]:
            delta[col] = float(b[col] - a[col])
        summary = pd.concat([summary, pd.DataFrame([delta])], ignore_index=True)

    return fold_metrics_df, fold_predictions_df, summary



In [ ]:
# ===============================
# Cell 3: Define Models + Grids
# ===============================
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
xgb_grid = {
    "n_estimators": [100],
    "max_depth": [3, 5],
    "learning_rate": [0.1],
    "scale_pos_weight": [1, (y == 0).sum() / (y == 1).sum()]
}

lr_model = LogisticRegression(solver="liblinear", class_weight="balanced")
lr_grid = {"C": [0.1, 1, 10]}

mlp_model = MLPClassifier(random_state=42, max_iter=500)
mlp_grid = {
    "hidden_layer_sizes": [(64,), (64, 32)],
    "alpha": [0.0001, 0.01]
}

In [ ]:
# ===============================
# Cell 4: Two Stage Analyses + Outputs
# ===============================
out_dir = "Results_two_step"
os.makedirs(out_dir, exist_ok=True)

models = {
    "XGBoost": (xgb_model, xgb_grid),
    "LogReg": (lr_model, lr_grid),
    "MLP": (mlp_model, mlp_grid)
}

# You can change this to match what you used elsewhere
primary_metrics = [
    ("F1 (macro)", "f1"),
    ("Balanced Acc", "balanced_accuracy"),
    ("AUCPR", "average_precision"),
    ("F1 Weighted", "f1_weighted"),
]

all_summaries = []
all_fold_metrics = []
all_fold_preds = []

for model_name, (base_model, base_grid) in models.items():
    for metric_name, metric_key in primary_metrics:
        print(f"\n🚀 Running TWO STAGE {model_name} optimized for {metric_name} ...")

        try:
            fold_metrics_df, fold_preds_df, summary_df = two_stage_nested_cv_evaluation(
                X=X,
                y=y,
                stage1_model=base_model,
                stage1_grid=base_grid,
                stage2_model=base_model,
                stage2_grid=base_grid,
                questionnaire_cols=questionnaire_cols,
                actigraphy_cols=actigraphy_cols,
                inner_cv=inner_cv,
                outer_cv=outer_cv,
                primary_metric=metric_key
            )

            fold_metrics_df.insert(0, "Model", f"{model_name} ({metric_name})")
            fold_preds_df.insert(0, "Model", f"{model_name} ({metric_name})")
            summary_df.insert(0, "Model", f"{model_name} ({metric_name})")

            all_fold_metrics.append(fold_metrics_df)
            all_fold_preds.append(fold_preds_df)
            all_summaries.append(summary_df)

            # Save per run
            safe_name = f"{model_name}_{metric_name}".replace(" ", "_").replace("(", "").replace(")", "")
            fold_metrics_df.to_csv(os.path.join(out_dir, f"two_stage_fold_metrics_{safe_name}.csv"), index=False)
            fold_preds_df.to_csv(os.path.join(out_dir, f"two_stage_fold_predictions_{safe_name}.csv"), index=False)
            summary_df.to_csv(os.path.join(out_dir, f"two_stage_summary_{safe_name}.csv"), index=False)

        except Exception as e:
            print(f"⚠️ Skipping {model_name} ({metric_name}) due to error: {e}")
            continue

# Combine outputs
if len(all_fold_metrics) > 0:
    all_fold_metrics_df = pd.concat(all_fold_metrics, ignore_index=True)
    all_fold_preds_df = pd.concat(all_fold_preds, ignore_index=True)
    all_summaries_df = pd.concat(all_summaries, ignore_index=True)

    all_fold_metrics_df.to_csv(os.path.join(out_dir, "ALL_two_stage_fold_metrics.csv"), index=False)
    all_fold_preds_df.to_csv(os.path.join(out_dir, "ALL_two_stage_fold_predictions.csv"), index=False)
    all_summaries_df.to_csv(os.path.join(out_dir, "ALL_two_stage_summary.csv"), index=False)

    print("\n✅ Two stage analyses complete!")
    print(f"📂 Folder: {out_dir}")
    print("📊 Combined summary → ALL_two_stage_summary.csv")
    print("📌 Key effect columns: FN and Recovered_FNs and Delta rows")

    display(all_summaries_df)
else:
    print("⚠️ No results were produced. Check for errors or too few positives in low-risk subsets.")

In [ ]:
# ===============================
# A) Are there any meaningful differences in actigraphy variables between recovered and non-recovered FN in stage 2
# ===============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu

# ---- Paths ----
pred_path = "Results_two_step/ALL_two_stage_fold_predictions.csv"
data_path = "../data/mcs_sh_sample_actig_Feb_28_preprocessed.csv"

# ---- Actigraphy variables (exactly yours) ----
actigraphy_cols = [
    "mean_acc_24h",
    "m5_hour_start",
    "l5_hour_start",
    "mvpa_acc_5sec"
]

# ---- Load fold predictions ----
preds = pd.read_csv(pred_path)

required_cols = [
    "Model", "Fold", "Index", "y_true",
    "p_stage1", "p_stage2",
    "pred_stage1", "pred_two_stage",
    "lowrisk_after_stage1"
]
missing = [c for c in required_cols if c not in preds.columns]
if len(missing) > 0:
    raise ValueError(f"Missing columns in predictions file: {missing}")

print("✅ Loaded predictions:", preds.shape)

# ---- Load dataset to attach actigraphy values by row index ----
df = pd.read_csv(data_path)
df["suicide_17y"] = df["suicide_17y"].map({"no": 0, "yes": 1})

X_full = df.drop(columns=["suicide_17y"])
if "id" in X_full.columns:
    X_full = X_full.drop(columns=["id"])

X_full = X_full.replace({"True": True, "False": False}).astype(float)
X_full = X_full.reset_index().rename(columns={"index": "Index"})

missing_actig = [c for c in actigraphy_cols if c not in X_full.columns]
if len(missing_actig) > 0:
    raise ValueError(f"These actigraphy cols are missing from dataset: {missing_actig}")

# Merge actigraphy columns into preds
preds = preds.merge(X_full[["Index"] + actigraphy_cols], on="Index", how="left")

print("✅ Merged actigraphy into predictions:", preds.shape)
print("Actigraphy missingness after merge:")
print(preds[actigraphy_cols].isna().mean())

# ---- Choose one model run to analyze ----
# Tip: run this line to see exact model names available
print("\nAvailable Model values:")
print(sorted(preds["Model"].unique())[:30])

MODEL_TO_CHECK = "LogReg (AUCPR)"  # <-- CHANGE THIS to match one of preds["Model"].unique()

df_m = preds[preds["Model"] == MODEL_TO_CHECK].copy()
if len(df_m) == 0:
    raise ValueError(f"No rows found for Model == {MODEL_TO_CHECK}. Check the exact name.")

print(f"\n✅ Using model: {MODEL_TO_CHECK}")
print("Rows for this model:", len(df_m))

# ---- Step 1: Identify Stage 1 false negatives on OUTER TEST sets ----
# FN under stage 1 means: y_true=1 and pred_stage1=0
df_fn = df_m[(df_m["y_true"] == 1) & (df_m["pred_stage1"] == 0)].copy()

print("\nStage 1 false negatives (outer-test predictions):", len(df_fn))

# ---- Step 2: Split into recovered vs not recovered by two-stage ----
# recovered means: stage 1 missed but two-stage predicts 1
df_fn["Recovered"] = (df_fn["pred_two_stage"] == 1).astype(int)

n_rec = int(df_fn["Recovered"].sum())
n_not = int((df_fn["Recovered"] == 0).sum())
print("Recovered by Stage 2:", n_rec)
print("Not recovered:", n_not)

if len(df_fn) < 5 or n_rec < 2 or n_not < 2:
    print("\n⚠️ Warning: too few cases in one of the groups for stable inference.")
    print("You can still inspect plots, but p-values will not be reliable.")

# ---- Step 3: Compare actigraphy distributions recovered vs not recovered ----
rows = []
for c in actigraphy_cols:
    g_rec = df_fn[df_fn["Recovered"] == 1][c].dropna().values
    g_not = df_fn[df_fn["Recovered"] == 0][c].dropna().values

    rec_mean = float(np.mean(g_rec)) if len(g_rec) > 0 else np.nan
    not_mean = float(np.mean(g_not)) if len(g_not) > 0 else np.nan
    rec_med = float(np.median(g_rec)) if len(g_rec) > 0 else np.nan
    not_med = float(np.median(g_not)) if len(g_not) > 0 else np.nan

    p_val = np.nan
    if len(g_rec) >= 3 and len(g_not) >= 3:
        try:
            _, p_val = mannwhitneyu(g_rec, g_not, alternative="two-sided")
            p_val = float(p_val)
        except Exception:
            p_val = np.nan

    rows.append({
        "Feature": c,
        "Recovered_n": int(len(g_rec)),
        "NotRecovered_n": int(len(g_not)),
        "Recovered_mean": rec_mean,
        "NotRecovered_mean": not_mean,
        "Recovered_median": rec_med,
        "NotRecovered_median": not_med,
        "p_value_MWU": p_val
    })

diff_table = pd.DataFrame(rows).sort_values("p_value_MWU", na_position="last").reset_index(drop=True)

print("\n=== Actigraphy differences among Stage 1 false negatives ===")
display(diff_table)

# ---- Boxplots for visual inspection ----
for c in actigraphy_cols:
    plt.figure()
    df_fn.boxplot(column=c, by="Recovered")
    plt.title(f"{MODEL_TO_CHECK}\n{c} among Stage 1 false negatives\n0 not recovered, 1 recovered")
    plt.suptitle("")
    plt.xlabel("Recovered")
    plt.ylabel(c)
    plt.show()

# ---- Recovery rate summary ----
fn_total = len(df_fn)
recovered = n_rec
recovery_rate = recovered / fn_total if fn_total > 0 else np.nan
print("\nFN recovery rate among Stage 1 false negatives:", round(recovery_rate, 3) if not np.isnan(recovery_rate) else "NA")